# LeetCode 2331 — Evaluate Boolean Binary Tree bằng Python

Link đề: https://leetcode.com/problems/evaluate-boolean-binary-tree/ · Độ khó: **Easy** · Chủ đề: Tree, DFS, BFS, Binary Tree

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def evaluateTree(self, root: Optional[TreeNode]) -> bool:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu **cây nhị phân** (binary tree), nút lá, nút trong, và tự dựng `TreeNode` cùng hàm `build_tree` từ danh sách kiểu LeetCode (`[2,1,3,None,None,0,1]`).
2. Nắm tư duy **đệ quy trên cây**: giá trị của một nút được tính từ giá trị của hai con (chia để trị / hậu thứ tự).
3. Biết cách **khử đệ quy** bằng ngăn xếp (stack) với duyệt hậu thứ tự, và cách đánh giá bằng **BFS đảo ngược** hoặc **biểu thức hậu tố (RPN)**.
4. Viết 5 cách: DFS đệ quy, DFS dùng bảng toán tử `operator`, DFS lặp bằng stack, BFS rồi tính ngược, và hậu tố + ngăn xếp; đối chiếu với `eval` trên biểu thức chuỗi.
5. Tránh các lỗi: quên điều kiện dừng ở lá, nhầm mã `2` là OR và `3` là AND, trả `int` thay vì `bool`, đệ quy chỉ một nhánh.

## 2. Đề bài & ví dụ

Cho gốc `root` của một **cây nhị phân đầy đủ** (*full binary tree*: mỗi nút có **0 hoặc 2** con) với các giá trị:

- **Nút lá** có giá trị `0` (**False**) hoặc `1` (**True**).
- **Nút trong** có giá trị `2` (phép **OR**) hoặc `3` (phép **AND**).

**Giá trị đánh giá** của một nút được định nghĩa:

- Nếu là lá: `False` nếu giá trị `0`, `True` nếu giá trị `1`.
- Nếu là nút trong: áp dụng phép toán của nút (`OR` hoặc `AND`) lên giá trị đánh giá của **hai con** của nó.

Trả về giá trị đánh giá của `root` (kiểu `bool`).

| Ví dụ | root (dạng danh sách) | Output | Giải thích |
|---|---|---|---|
| 1 | `[2,1,3,null,null,0,1]` | `True` | `OR(1, AND(0,1))` = `OR(True, False)` = `True` |
| 2 | `[0]` | `False` | chỉ có một nút lá `0` |

Constraints: số nút trong `[1, 1000]`; `0 <= Node.val <= 3`; mỗi nút có `0` hoặc `2` con; nút lá có `Node.val` là `0` hoặc `1`; nút trong có `Node.val` là `2` hoặc `3`.

In [1]:
from collections import deque
from typing import List, Optional

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

def build_tree(values):
    # Dựng cây từ danh sách theo thứ tự tầng (level order) kiểu LeetCode, None = không có nút.
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root

def to_expr(node):
    # Biểu thức Python của cây, chỉ dùng để quan sát.
    if node.left is None and node.right is None:
        return "True" if node.val == 1 else "False"
    op = "or" if node.val == 2 else "and"
    return f"({to_expr(node.left)} {op} {to_expr(node.right)})"

examples = [([2, 1, 3, None, None, 0, 1], True), ([0], False)]
for values, expected in examples:
    root = build_tree(values)
    print(values, "->", to_expr(root), "=", eval(to_expr(root)), "; mong đợi", expected)

[2, 1, 3, None, None, 0, 1] -> (True or (False and True)) = True ; mong đợi True
[0] -> False = False ; mong đợi False


## 3. Kiến thức nền

### 3.1. Cây nhị phân và nút `TreeNode`

Mỗi nút lưu `val` và hai con `left`, `right` (có thể là `None`). **Nút lá** không có con. **Cây đầy đủ** (full): mỗi nút hoặc là lá, hoặc có **đúng hai** con — nên nếu `node.left` tồn tại thì `node.right` cũng tồn tại. LeetCode đã định nghĩa sẵn `TreeNode`; ở đây ta tự khai báo để chạy thử.

In [2]:
leaf1 = TreeNode(1)
leaf0 = TreeNode(0)
root = TreeNode(3, leaf0, leaf1)           # nút AND có hai lá 0 và 1
print(root.val, root.left.val, root.right.val, root.left.left)

3 0 1 None


### 3.2. Danh sách kiểu LeetCode và hàm `build_tree`

`[2,1,3,None,None,0,1]` liệt kê các nút **theo từng tầng, từ trái sang phải**. Quy tắc: lần lượt lấy một nút `node` ra khỏi hàng đợi, hai giá trị kế tiếp của danh sách là con trái, con phải của `node` (`None` nghĩa là vắng). Cây ví dụ 1:

```
      2
     / \
    1   3
       / \
      0   1
```
Hàm `build_tree` ở trên dùng `collections.deque` làm hàng đợi (BFS).

In [3]:
def preorder(node):
    return [] if node is None else [node.val] + preorder(node.left) + preorder(node.right)

t = build_tree([2, 1, 3, None, None, 0, 1])
print(preorder(t))
print(preorder(build_tree([0])), preorder(build_tree([])))

[2, 1, 3, 0, 1]
[0] []


### 3.3. Đệ quy trên cây

Nguyên tắc: một hàm `f(node)` giả định `f(node.left)` và `f(node.right)` **đã đúng**, rồi ghép lại để ra `f(node)`. Cần **điều kiện dừng** (ở lá hoặc nút rỗng). Với cây này lá luôn là `0/1` còn nút trong luôn có đủ hai con.

In [4]:
def count_nodes(node):
    if node is None:
        return 0
    return 1 + count_nodes(node.left) + count_nodes(node.right)

def height(node):
    if node is None:
        return 0
    return 1 + max(height(node.left), height(node.right))

t = build_tree([2, 1, 3, None, None, 0, 1])
print(count_nodes(t), height(t))

5 3


### 3.4. Phép OR/AND của Python và module `operator`

`a or b`, `a and b` trên `bool`. Module `operator` có `operator.or_` và `operator.and_` — thực ra là toán tử bit `|` và `&`; với `bool` chúng cho kết quả `bool` đúng. Ta có thể lập bảng ánh xạ `{2: or, 3: and}`.

In [5]:
import operator
OPS = {2: operator.or_, 3: operator.and_}
print(OPS[2](True, False), OPS[3](True, False), OPS[3](True, True))
print(True | False, True & False, type(True | False))

True False True
True False <class 'bool'>


### 3.5. Ngăn xếp (stack) và hàng đợi (queue)

- Stack (LIFO): `list.append` / `list.pop()` — vào sau ra trước; dùng để **khử đệ quy** (DFS lặp).
- Queue (FIFO): `collections.deque` với `append` / `popleft()` — dùng cho **BFS** (duyệt theo tầng).

In [6]:
stack = []
stack.append(1); stack.append(2); stack.append(3)
print(stack.pop(), stack.pop(), stack)
from collections import deque
q = deque([1, 2, 3])
print(q.popleft(), q.popleft(), list(q))

3 2 [1]
1 2 [3]


### 3.6. Hậu thứ tự (postorder) và biểu thức hậu tố (RPN)

Duyệt **hậu thứ tự**: con trái → con phải → nút. Với cây biểu thức, hậu thứ tự chính là **biểu thức hậu tố** (Reverse Polish Notation): toán hạng trước, toán tử sau. Đánh giá RPN bằng stack: gặp toán hạng thì đẩy vào; gặp toán tử thì lấy hai giá trị ra, tính, đẩy kết quả vào.

In [7]:
def postorder(node):
    if node is None:
        return []
    return postorder(node.left) + postorder(node.right) + [node.val]

t = build_tree([2, 1, 3, None, None, 0, 1])
print(postorder(t))      # 1 0 1 3 2 : toán hạng 1, rồi (0 1 AND), rồi OR

[1, 0, 1, 3, 2]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Tính tay ví dụ 1 từ **dưới lên**:

```
      2 (OR)
     /      \
    1       3 (AND)
           /      \
          0        1
```

- Nút `AND` có hai lá `0` và `1`: `False and True = False`.
- Nút gốc `OR` có hai con: lá `1` (`True`) và nút `AND` vừa tính (`False`): `True or False = True`.

Vậy giá trị của một nút **phụ thuộc vào giá trị hai con** → cần tính con trước, rồi tính cha. Đó là **hậu thứ tự**: *đi xuống tận lá, rồi gộp kết quả đi lên*.

In [8]:
root = build_tree([2, 1, 3, None, None, 0, 1])
and_node = root.right
print("AND:", bool(and_node.left.val) and bool(and_node.right.val))
print("gốc OR:", bool(root.left.val) or (bool(and_node.left.val) and bool(and_node.right.val)))

AND: False
gốc OR: True


## 5. Cách 1 — Brute Force / baseline: DFS đệ quy trực tiếp

Định nghĩa đệ quy bám sát đề: lá thì trả `val == 1`; nút `2` thì `OR` hai con, nút `3` thì `AND` hai con.

In [9]:
def eval_recursive(node):
    if node.left is None and node.right is None:     # lá
        return node.val == 1
    left = eval_recursive(node.left)
    right = eval_recursive(node.right)
    if node.val == 2:
        return left or right
    return left and right

print(eval_recursive(build_tree([2, 1, 3, None, None, 0, 1])), eval_recursive(build_tree([0])))

True False


In [10]:
from typing import List, Optional

class SolutionRecursive:
    def evaluateTree(self, root: Optional[TreeNode]) -> bool:
        if root.left is None and root.right is None:
            return root.val == 1
        left = self.evaluateTree(root.left)
        right = self.evaluateTree(root.right)
        if root.val == 2:
            return left or right
        return left and right

print(SolutionRecursive().evaluateTree(build_tree([2, 1, 3, None, None, 0, 1])))
print(SolutionRecursive().evaluateTree(build_tree([0])))

True
False


**Độ phức tạp:** `n` là số nút. Mỗi nút được thăm đúng một lần với công việc `O(1)` → **Time `O(n)`**. Ngăn xếp đệ quy sâu bằng **chiều cao** `h` → **Space `O(h)`** (`O(log n)` nếu cân bằng, tệ nhất `O(n)` nếu lệch).

**Nút thắt:** gần như tối ưu; cải tiến chỉ là **rút gọn mã** (bảng toán tử), **tránh đệ quy sâu** (stack/BFS), hoặc **dừng sớm** (short-circuit).

## 6. Ý tưởng thuật toán / bước đột phá

- Giá trị của nút = `phép_toán(giá_trị_con_trái, giá_trị_con_phải)` → công thức đệ quy tự nhiên.
- Vì cây **đầy đủ**, chỉ cần kiểm tra "là lá" bằng một con (`root.left is None`). Không cần lo trường hợp thiếu một con.
- **Short-circuit:** với `OR`, nếu con trái đã `True` thì khỏi tính con phải; với `AND`, nếu con trái `False` thì khỏi tính con phải. Dùng `or`/`and` của Python (đánh giá lười) trực tiếp trên lời gọi đệ quy.
- **Khử đệ quy:** mọi thứ cần là "con xong trước cha": hậu thứ tự lặp, hoặc BFS rồi xử lý theo thứ tự **ngược** của BFS (cha xuất hiện trước con trong BFS, nên đảo ngược thì con trước cha).
- Hậu thứ tự = biểu thức hậu tố → đánh giá bằng một stack giá trị.

## 7. Các cách tiếp theo

### Cách 2 — DFS đệ quy ngắn gọn (short-circuit)

Dùng chính toán tử `or`/`and` của Python, vốn đánh giá **lười**: vế phải chỉ được gọi khi cần.

In [11]:
from typing import List, Optional

class SolutionShort:
    def evaluateTree(self, root: Optional[TreeNode]) -> bool:
        if root.left is None:                       # cây đầy đủ: không có con trái thì là lá
            return root.val == 1
        if root.val == 2:
            return self.evaluateTree(root.left) or self.evaluateTree(root.right)
        return self.evaluateTree(root.left) and self.evaluateTree(root.right)

print(SolutionShort().evaluateTree(build_tree([2, 1, 3, None, None, 0, 1])))
print(SolutionShort().evaluateTree(build_tree([0])))

True
False


**Độ phức tạp:** Time `O(n)` trong trường hợp xấu nhất (vẫn phải thăm hết khi không cắt được), tốt nhất có thể ít hơn nhờ short-circuit; Space `O(h)`.

### Cách 3 — DFS lặp bằng ngăn xếp (hậu thứ tự)

Để không bị giới hạn độ sâu đệ quy, ta dùng `stack` các nút và từ điển `value` lưu kết quả của từng nút theo `id(node)` (hoặc trực tiếp ghi vào nút). Kỹ thuật "hai lần thăm": lần đầu đẩy hai con vào, lần sau (khi con đã xong) tính cho cha.

In [12]:
from typing import List, Optional

class SolutionIterative:
    def evaluateTree(self, root: Optional[TreeNode]) -> bool:
        value = {}                                   # id(node) -> bool
        stack = [(root, False)]                      # (nút, đã xử lý con chưa)
        while stack:
            node, children_done = stack.pop()
            if node.left is None:                    # lá
                value[id(node)] = node.val == 1
            elif children_done:
                l, r = value[id(node.left)], value[id(node.right)]
                value[id(node)] = (l or r) if node.val == 2 else (l and r)
            else:
                stack.append((node, True))           # quay lại sau
                stack.append((node.left, False))
                stack.append((node.right, False))
        return value[id(root)]

print(SolutionIterative().evaluateTree(build_tree([2, 1, 3, None, None, 0, 1])))
print(SolutionIterative().evaluateTree(build_tree([0])))

True
False


**Độ phức tạp:** Time `O(n)`; Space `O(n)` cho từ điển `value` (có thể giảm xuống `O(h)` nếu xoá giá trị con sau khi dùng).

### Cách 4 — BFS lấy mọi nút, rồi tính theo thứ tự ngược

BFS cho danh sách nút theo tầng: cha luôn đứng **trước** con của nó. Duyệt danh sách đó **từ cuối về đầu**, mỗi lần gặp nút nào thì hai con của nó đã được tính xong. Không cần đệ quy.

In [13]:
from collections import deque
from typing import List, Optional

class SolutionBFS:
    def evaluateTree(self, root: Optional[TreeNode]) -> bool:
        order = []
        queue = deque([root])
        while queue:
            node = queue.popleft()
            order.append(node)
            if node.left is not None:
                queue.append(node.left)
                queue.append(node.right)
        value = {}
        for node in reversed(order):                 # con trước cha
            if node.left is None:
                value[id(node)] = node.val == 1
            else:
                l, r = value[id(node.left)], value[id(node.right)]
                value[id(node)] = (l or r) if node.val == 2 else (l and r)
        return value[id(root)]

print(SolutionBFS().evaluateTree(build_tree([2, 1, 3, None, None, 0, 1])))
print(SolutionBFS().evaluateTree(build_tree([0])))

True
False


**Độ phức tạp:** Time `O(n)`, Space `O(n)` (danh sách `order`, từ điển `value`, hàng đợi rộng nhất bằng chiều rộng cây).

### Cách 5 — Hậu tố (RPN) + ngăn xếp giá trị

Sinh dãy hậu thứ tự của cây (toán hạng trước, toán tử sau), rồi đánh giá bằng stack: gặp lá `0/1` thì đẩy `False/True`; gặp `2`/`3` thì lấy hai giá trị trên cùng, tính OR/AND và đẩy kết quả. Cuối cùng stack còn đúng **một** giá trị.

In [14]:
from typing import List, Optional

class SolutionRPN:
    def evaluateTree(self, root: Optional[TreeNode]) -> bool:
        # 1) hậu thứ tự lặp: dùng hai stack rồi đảo ngược (gốc-phải-trái, sau đó đảo)
        out = []
        stack = [root]
        while stack:
            node = stack.pop()
            out.append(node)
            if node.left is not None:
                stack.append(node.left)
                stack.append(node.right)
        out.reverse()                                # hậu thứ tự: trái, phải, nút
        # 2) đánh giá hậu tố
        values = []
        for node in out:
            if node.left is None:
                values.append(node.val == 1)
            else:
                b, a = values.pop(), values.pop()
                values.append((a or b) if node.val == 2 else (a and b))
        return values[0]

print(SolutionRPN().evaluateTree(build_tree([2, 1, 3, None, None, 0, 1])))
print(SolutionRPN().evaluateTree(build_tree([0])))

True
False


**Độ phức tạp:** Time `O(n)`, Space `O(n)`. Ghi chú: ở bước 1 ta lấy thứ tự "nút, phải, trái" rồi **đảo** thành "trái, phải, nút" — đúng hậu thứ tự.

## 8. TRACE từng bước

Trace DFS đệ quy (Cách 1) trên cây `[2,1,3,null,null,0,1]`. Cột thụt lề cho biết độ sâu:

| gọi | nút | loại | kết quả |
|---|---|---|---|
| `f(2)` | gốc OR | nút trong | đợi hai con |
| `f(1)` | lá | lá | `True` |
| `f(3)` | AND | nút trong | đợi hai con |
| `f(0)` | lá | lá | `False` |
| `f(1)` | lá | lá | `True` |
| `f(3)` trả về | AND | `False and True` | `False` |
| `f(2)` trả về | OR | `True or False` | `True` |

In [15]:
def trace_eval(node, depth=0):
    pad = "  " * depth
    if node.left is None:
        result = node.val == 1
        print(f"{pad}lá {node.val} -> {result}")
        return result
    op = "OR" if node.val == 2 else "AND"
    print(f"{pad}nút {node.val} ({op}) bắt đầu")
    l = trace_eval(node.left, depth + 1)
    r = trace_eval(node.right, depth + 1)
    result = (l or r) if node.val == 2 else (l and r)
    print(f"{pad}nút {node.val} ({op}): {l} {op.lower()} {r} -> {result}")
    return result

print("kết quả:", trace_eval(build_tree([2, 1, 3, None, None, 0, 1])))

nút 2 (OR) bắt đầu
  lá 1 -> True
  nút 3 (AND) bắt đầu
    lá 0 -> False
    lá 1 -> True
  nút 3 (AND): False and True -> False
nút 2 (OR): True or False -> True
kết quả: True


In [16]:
def trace_rpn(values):
    root = build_tree(values)
    out, stack = [], [root]
    while stack:
        node = stack.pop()
        out.append(node)
        if node.left is not None:
            stack.append(node.left)
            stack.append(node.right)
    out.reverse()
    print("dãy hậu tố:", [n.val for n in out])
    st = []
    for node in out:
        if node.left is None:
            st.append(node.val == 1)
        else:
            b, a = st.pop(), st.pop()
            st.append((a or b) if node.val == 2 else (a and b))
        print(f"  xử lý {node.val}: stack = {st}")
    return st[0]

print("kết quả:", trace_rpn([2, 1, 3, None, None, 0, 1]))

dãy hậu tố: [1, 0, 1, 3, 2]
  xử lý 1: stack = [True]
  xử lý 0: stack = [True, False]
  xử lý 1: stack = [True, False, True]
  xử lý 3: stack = [True, False]
  xử lý 2: stack = [True]
kết quả: True


## 9. Code nộp LeetCode

Trên LeetCode, `TreeNode` đã được cung cấp sẵn (chỉ nằm trong comment); cell này giữ ngay các `import` cần thiết và dùng `TreeNode` đã khai báo ở trên của notebook.

In [17]:
from typing import List, Optional

class Solution:
    def evaluateTree(self, root: Optional[TreeNode]) -> bool:
        if root.left is None:                  # cây đầy đủ: lá không có con
            return root.val == 1
        if root.val == 2:
            return self.evaluateTree(root.left) or self.evaluateTree(root.right)
        return self.evaluateTree(root.left) and self.evaluateTree(root.right)

print(Solution().evaluateTree(build_tree([2, 1, 3, None, None, 0, 1])))
print(Solution().evaluateTree(build_tree([0])))

True
False


## 10. Vì sao thuật toán đúng

- **Quy nạp theo cấu trúc cây.** *Cơ sở:* nếu nút là lá thì giá trị đánh giá theo đề là `False` khi `val = 0`, `True` khi `val = 1`, và `root.val == 1` cho đúng kết quả đó. *Bước quy nạp:* nút trong có đúng hai con, mà các con thuộc cây con nhỏ hơn nên (theo giả thiết quy nạp) `f` trả đúng giá trị đánh giá của chúng. Nút `2` → `OR`, nút `3` → `AND`, đúng định nghĩa; vậy `f(nút)` cũng đúng.
- **Cây đầy đủ** đảm bảo `node.left is None` ⇔ `node.right is None` ⇔ lá, nên điều kiện dừng không bỏ sót trường hợp "chỉ một con".
- **Short-circuit** không đổi kết quả: `True or x = True` và `False and x = False` bất kể `x`; cây không có tác dụng phụ.
- **Cách lặp và BFS:** invariant "khi tính giá trị một nút, giá trị hai con đã có trong `value`". Hậu thứ tự lặp và BFS đảo ngược đều đảm bảo thứ tự "con trước cha".
- **RPN:** mỗi nút trong lấy đúng hai giá trị trên cùng của stack — chính là kết quả hai cây con vừa được xử lý ngay trước nó trong hậu thứ tự; cuối cùng chỉ còn giá trị của gốc.

## 11. Phân tích Time / Space Complexity

`n` = số nút (`≤ 1000`), `h` = chiều cao cây.

| Cách | Time | Space |
|---|---|---|
| 1. DFS đệ quy | `O(n)` | `O(h)` (ngăn xếp gọi hàm) |
| 2. DFS short-circuit | `O(n)` tệ nhất, có thể ít hơn | `O(h)` |
| 3. DFS lặp (stack) | `O(n)` | `O(n)` (từ điển giá trị) |
| 4. BFS rồi tính ngược | `O(n)` | `O(n)` |
| 5. Hậu tố + stack giá trị | `O(n)` | `O(n)` |

Mỗi nút được thăm một số hằng lần. Trường hợp cây lệch, `h ≈ n/2` (vì cây đầy đủ), nên với `n = 1000` độ sâu đệ quy tối đa khoảng 500 — vẫn dưới giới hạn đệ quy mặc định 1000 của Python.

## 12. Bộ test

Ví dụ đề, edge case (một lá `0` / `1`, cây lệch rất sâu, toàn `AND`, toàn `OR`), kiểm tra `build_tree` bằng chuyển đổi ngược, và test ngẫu nhiên trên **cây đầy đủ ngẫu nhiên** đối chiếu **tất cả cách** và `eval` biểu thức.

In [18]:
import random

solvers = [SolutionRecursive(), SolutionShort(), SolutionIterative(), SolutionBFS(), SolutionRPN(), Solution()]

def to_list(root):
    # chuyển cây về danh sách level order kiểu LeetCode (bỏ None thừa ở cuối)
    if root is None:
        return []
    out, q = [], deque([root])
    while q:
        node = q.popleft()
        if node is None:
            out.append(None)
            continue
        out.append(node.val)
        q.append(node.left)
        q.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out

# build_tree <-> to_list khớp nhau
for lst in ([2, 1, 3, None, None, 0, 1], [0], [1], [3, 1, 1], [2, 3, 2, 1, 0, 0, 1]):
    assert to_list(build_tree(lst)) == lst, lst
print("build_tree/to_list: OK")

fixed = [
    ([2, 1, 3, None, None, 0, 1], True),
    ([0], False),
    ([1], True),
    ([2, 0, 0], False),                       # OR(F, F)
    ([2, 0, 1], True),                        # OR(F, T)
    ([3, 1, 1], True),                        # AND(T, T)
    ([3, 1, 0], False),                       # AND(T, F)
    ([3, 2, 2, 1, 0, 0, 0], False),           # AND(OR(T,F), OR(F,F))
    ([2, 3, 3, 1, 0, 1, 1], True),            # OR(AND(T,F), AND(T,T))
    ([2, 3, 2, 0, 1, 0, 0], False),           # OR(AND(F,T), OR(F,F))
]
for values, expected in fixed:
    root = build_tree(values)
    for sol in solvers:
        assert sol.evaluateTree(root) is expected, (type(sol).__name__, values, expected)
print("test cố định: OK")

def make_full_tree(rng, internal):
    # cây đầy đủ ngẫu nhiên với `internal` nút trong (=> internal + 1 lá)
    if internal == 0:
        return TreeNode(rng.randint(0, 1))
    left_internal = rng.randint(0, internal - 1)
    return TreeNode(rng.randint(2, 3), make_full_tree(rng, left_internal), make_full_tree(rng, internal - 1 - left_internal))

rng = random.Random(2331)
true_count = 0
for _ in range(600):
    root = make_full_tree(rng, rng.randint(0, 25))
    expected = eval(to_expr(root))
    true_count += expected
    for sol in solvers:
        assert sol.evaluateTree(root) is expected, (type(sol).__name__, to_list(root))
print("600 test ngẫu nhiên: OK, số lần True:", true_count)
assert 0 < true_count < 600

# cây lệch sâu: chuỗi AND/OR dài, mỗi nút trong có một lá bên phải
def make_chain(depth, op, leaf_chain, leaf_end):
    node = TreeNode(leaf_end)
    for _ in range(depth):
        node = TreeNode(op, node, TreeNode(leaf_chain))
    return node

deep_and = make_chain(400, 3, 1, 1)           # AND toàn True -> True
deep_or = make_chain(400, 2, 0, 0)            # OR toàn False -> False
deep_mix = make_chain(400, 3, 1, 0)           # AND với lá cuối False -> False
for tree, expected in [(deep_and, True), (deep_or, False), (deep_mix, False)]:
    for sol in solvers:
        assert sol.evaluateTree(tree) is expected, (type(sol).__name__, expected)
print("cây lệch sâu (801 nút): OK")

build_tree/to_list: OK
test cố định: OK
600 test ngẫu nhiên: OK, số lần True: 315
cây lệch sâu (801 nút): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Nhầm ý nghĩa mã `2` và `3`

Đề: `2` là **OR**, `3` là **AND**. Dễ nhầm vì nghĩ "3 nhiều hơn nên là OR".

In [19]:
def wrong1(node):
    if node.left is None:
        return node.val == 1
    l, r = wrong1(node.left), wrong1(node.right)
    return (l or r) if node.val == 3 else (l and r)       # đảo ngược

t = build_tree([2, 0, 1])                                  # OR(False, True) = True
print("sai :", wrong1(t), "(đáng lẽ True)")
print("đúng:", Solution().evaluateTree(t))

sai : False (đáng lẽ True)
đúng: True


### Lỗi 2: Thiếu điều kiện dừng ở nút lá

Không kiểm tra `left is None` thì truy cập `None.val` → `AttributeError`.

In [20]:
def wrong2(node):
    l = wrong2(node.left)          # lá: node.left là None -> lỗi ở lần gọi sau
    r = wrong2(node.right)
    return (l or r) if node.val == 2 else (l and r)

try:
    wrong2(build_tree([2, 0, 1]))
except AttributeError as e:
    print("sai -> AttributeError:", e)
print("đúng:", Solution().evaluateTree(build_tree([2, 0, 1])))

sai -> AttributeError: 'NoneType' object has no attribute 'left'
đúng: True


### Lỗi 3: Trả về số `0/1` thay vì `bool`

Hàm phải trả `bool`. Trả `root.val` (số nguyên `1`/`0`) cho lá cho kiểu dữ liệu sai: `1` không phải `True`, và phép kiểm tra `is True` sẽ thất bại.

In [21]:
def wrong3(node):
    if node.left is None:
        return node.val                    # sai kiểu: int thay vì bool
    l, r = wrong3(node.left), wrong3(node.right)
    return (l or r) if node.val == 2 else (l and r)

t = build_tree([1])
print("sai :", repr(wrong3(t)), "| kiểu:", type(wrong3(t)).__name__)
print("đúng:", repr(Solution().evaluateTree(t)), "| kiểu:", type(Solution().evaluateTree(t)).__name__)

sai : 1 | kiểu: int
đúng: True | kiểu: bool


### Lỗi 4: Chỉ đệ quy một nhánh

Quên con còn lại cho kết quả sai ở cây có nhiều lá.

In [22]:
def wrong4(node):
    if node.left is None:
        return node.val == 1
    return wrong4(node.left)               # bỏ qua node.right và phép toán

t = build_tree([3, 1, 0])                  # AND(True, False) = False
print("sai :", wrong4(t), "(đáng lẽ False)")
print("đúng:", Solution().evaluateTree(t))

sai : True (đáng lẽ False)
đúng: False


### Lỗi 5: Quên `return` ở nhánh đệ quy

Tính ra `l or r` nhưng không `return` thì hàm trả về `None`.

In [23]:
def wrong5(node):
    if node.left is None:
        return node.val == 1
    l, r = wrong5(node.left), wrong5(node.right)
    (l or r) if node.val == 2 else (l and r)       # thiếu return

t = build_tree([2, 1, 3, None, None, 0, 1])
print("sai :", wrong5(t), "(None, đáng lẽ True)")
print("đúng:", Solution().evaluateTree(t))

sai : None (None, đáng lẽ True)
đúng: True


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Giá trị của nút phụ thuộc giá trị hai con" → **đệ quy hậu thứ tự** (như LeetCode 104 Maximum Depth, 110 Balanced Tree, 112 Path Sum).
- Cây **đầy đủ** (mỗi nút 0 hoặc 2 con): kiểm tra lá bằng `node.left is None` là đủ.
- Biểu thức, toán tử, hai toán hạng → **cây biểu thức**; hậu thứ tự = hậu tố (RPN).
- Đệ quy quá sâu → chuyển sang **stack** hoặc **BFS rồi duyệt ngược**.
- Tự dựng `build_tree` từ danh sách `[2,1,3,None,None,0,1]` để chạy thử nhanh, đừng dựng tay từng nút.
- `or`/`and` của Python đánh giá lười — vừa gọn vừa nhanh.

## 15. Biến thể của bài

1. Có thêm phép **XOR / NOT** (nút một con) — không còn là cây đầy đủ.
2. Trả về **số nút** thực sự được tính khi dùng short-circuit.
3. LeetCode 1612 Check If Two Expression Trees are Equivalent, 1628 Design an Expression Tree With Evaluate Function (Premium).
4. LeetCode 104 Maximum Depth of Binary Tree, 100 Same Tree, 226 Invert Binary Tree: các bài đệ quy cây cơ bản.
5. Đếm **số cách đặt dấu ngoặc** để biểu thức đúng (bài DP kinh điển *Boolean Parenthesization*).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. DFS đệ quy | gộp giá trị hai con | `O(n)` | `O(h)` | rõ ràng nhất |
| 2. DFS short-circuit | dùng `or`/`and` lười | `O(n)` | `O(h)` | **khuyến nghị khi nộp**, gọn |
| 3. DFS lặp + stack | hậu thứ tự hai lần thăm | `O(n)` | `O(n)` | cây rất sâu |
| 4. BFS + tính ngược | con trước cha | `O(n)` | `O(n)` | tránh đệ quy, nghĩ theo tầng |
| 5. Hậu tố + stack giá trị | RPN | `O(n)` | `O(n)` | hiểu cây biểu thức ↔ RPN |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** `print(True or False and False, (True or False) and False, 1 == True, 2 | 1)` in ra gì?

**Bài 2 (tính tay).** Cây `[3, 2, 2, 1, 0, 0, 1]`: dựng tay rồi đánh giá. (Gốc `3` AND, hai con `2` OR.)

**Bài 3 (điền TODO).** Hoàn thành hàm đếm số **nút lá** của cây.

In [24]:
def count_leaves(node):
    # TODO: nếu node là None -> 0; nếu là lá (cả hai con None) -> 1; ngược lại cộng hai nhánh
    pass

# chưa gọi hàm ở đây để notebook chạy không lỗi

**Bài 4 (tự viết lại).** Viết `to_expression(root)` trả về chuỗi biểu thức đầy đủ ngoặc, ví dụ `[2,1,3,None,None,0,1]` → `"(True or (False and True))"`. Lá `1` thành `True`, lá `0` thành `False`; hãy tự viết lại mà không nhìn hàm `to_expr` ở đầu notebook.

In [25]:
def to_expression(root):
    # TODO: lá -> "True"/"False"; nút trong -> f"({trái} {op} {phải})"
    pass

**Bài 5 (mini challenge).** Viết `min_leaves_to_flip(root)`: số **lá tối thiểu** phải đổi giá trị (`0 ↔ 1`) để kết quả của cây trở thành `True`. Gợi ý: trả về cặp `(cost_to_true, cost_to_false)` cho mỗi nút (DP trên cây).

In [26]:
def min_leaves_to_flip(root):
    # TODO: DP trên cây: với lá giá trị v: cost_true = 0 nếu v==1 else 1; cost_false = 0 nếu v==0 else 1
    pass

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `and` ưu tiên cao hơn `or` → `True or (False and False)` = `True`; `(True or False) and False` = `False`; `1 == True` là `True`; `2 | 1 = 3` (OR bit).
- **Mức 2 (vừa):** Bài 2: `OR(1,0) = True`, `OR(0,1) = True`, gốc `AND(True, True) = True`. Bài 3: `if node is None: return 0`; `if node.left is None and node.right is None: return 1`.
- **Mức 3 (sâu):** Bài 5: với nút `OR`: `to_true = min(l.true, r.true)`, `to_false = l.false + r.false`; với nút `AND`: `to_true = l.true + r.true`, `to_false = min(l.false, r.false)`. Kết quả `cost_to_true` của gốc.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [27]:
# Bài 1 và 2
assert (True or False and False, (True or False) and False, 1 == True, 2 | 1) == (True, False, True, 3)
assert Solution().evaluateTree(build_tree([3, 2, 2, 1, 0, 0, 1])) is True

# Bài 3
def count_leaves(node):
    if node is None:
        return 0
    if node.left is None and node.right is None:
        return 1
    return count_leaves(node.left) + count_leaves(node.right)

# Bài 4
def to_expression(root):
    if root.left is None and root.right is None:
        return "True" if root.val == 1 else "False"
    op = "or" if root.val == 2 else "and"
    return f"({to_expression(root.left)} {op} {to_expression(root.right)})"

# Bài 5
def min_leaves_to_flip(root):
    def go(node):
        if node.left is None:
            return (0 if node.val == 1 else 1, 0 if node.val == 0 else 1)   # (to_true, to_false)
        lt, lf = go(node.left)
        rt, rf = go(node.right)
        if node.val == 2:                       # OR
            return (min(lt, rt), lf + rf)
        return (lt + rt, min(lf, rf))           # AND
    return go(root)[0]

assert count_leaves(build_tree([2, 1, 3, None, None, 0, 1])) == 3
assert count_leaves(build_tree([0])) == 1
assert count_leaves(build_tree([])) == 0
assert to_expression(build_tree([2, 1, 3, None, None, 0, 1])) == "(True or (False and True))"
assert to_expression(build_tree([0])) == "False"
assert min_leaves_to_flip(build_tree([2, 1, 3, None, None, 0, 1])) == 0       # đã True
assert min_leaves_to_flip(build_tree([3, 1, 0])) == 1                           # đổi lá 0 -> 1
assert min_leaves_to_flip(build_tree([3, 0, 0])) == 2
assert min_leaves_to_flip(build_tree([2, 0, 0])) == 1
assert min_leaves_to_flip(build_tree([0])) == 1
print("tất cả bài tập OK")

tất cả bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (đệ quy hậu thứ tự trên cây đầy đủ)

```python
class Solution:
    def evaluateTree(self, root: Optional[TreeNode]) -> bool:
        if root.left is None:              # lá
            return root.val == 1
        if root.val == 2:                  # OR
            return self.evaluateTree(root.left) or self.evaluateTree(root.right)
        return self.evaluateTree(root.left) and self.evaluateTree(root.right)   # AND (3)
```

**Checklist**

- [ ] `2` = OR, `3` = AND; `1` = True, `0` = False (đừng đảo ngược)?
- [ ] Có điều kiện dừng ở **lá** (`left is None`) trước khi truy cập con?
- [ ] Đệ quy **cả hai** nhánh và ghép đúng phép toán?
- [ ] Trả về `bool` (so sánh `root.val == 1`), không trả `int`?
- [ ] Cây một nút (`[0]`, `[1]`) được xử lý?
- [ ] Cây lệch sâu (~500 mức) vẫn chạy trong giới hạn đệ quy; nếu lo, dùng BFS/stack?
- [ ] Signature đúng `evaluateTree(self, root: Optional[TreeNode]) -> bool` và có `from typing import List, Optional`?